# Anime Recommender System

This notebook presents an anime recommender system that leverages both collaborative filtering and content-based filtering techniques. The recommender system aims to provide personalized anime recommendations based on user preferences and similarities between anime titles.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
ratings_path = '/kaggle/input/anime-recommendation-database-2020/animelist.csv'
data_path = '/kaggle/input/anime-recommendation-database-2020/anime.csv'

In [ ]:
anime_ratings = pd.read_csv(ratings_path)
anime_data = pd.read_csv(data_path)
anime_data.head()

In [ ]:
anime_data.info()

In [ ]:
anime_data = anime_data[['MAL_ID', 'Name', 'Score', 'Genres', 'Type', 'Episodes', 'Members']]
anime_data.rename(columns={'MAL_ID':"anime_id"}, inplace=True)
anime_data.info

In [ ]:
anime_ratings = anime_ratings.sample(frac=0.2)
anime_ratings.info()

In [ ]:
anime_ratings.anime_id.nunique()

In [ ]:
anime_complete = pd.merge(anime_data, anime_ratings, on='anime_id')
anime_complete = anime_complete.rename(columns={'rating': 'user_rating', 'Score': 'total_rating', "Name":'anime_title'})
anime_complete.info()

In [ ]:
anime_complete.isna().sum()

# Exploratory Data Analysis

This visualization allows us to easily identify the top 10 anime titles based on the frequency of user ratings. The height of each bar represents the user rating count for the corresponding anime title.

In [ ]:
top_10_anime = anime_complete['anime_title'].value_counts().nlargest(10)
palette = sns.color_palette('rocket', len(top_10_anime))

plt.bar(top_10_anime.index, top_10_anime.values, color=palette)

plt.title("Top 10 Anime by User Rating Count")
plt.xlabel("Anime Name")
plt.ylabel("User Rating Count")

plt.xticks(rotation=40, ha="right")

plt.show()

This visualization allows us to easily identify the top 10 anime titles with the highest number of members. The height of each bar represents the corresponding number of members for each anime title

In [ ]:
top_10_anime = anime_complete.sort_values(by="Members", ascending=False).drop_duplicates(subset='anime_title').head(10)

palette = sns.color_palette('rocket', len(top_10_anime))

plt.bar(top_10_anime['anime_title'], top_10_anime['Members'], color=palette)

plt.title("Top 10 Anime by Number of Members")
plt.xlabel("Anime Title")
plt.ylabel("Number of Members")

plt.xticks(rotation=40, ha='right')

plt.show()

In [ ]:
anime_features = anime_complete.copy()
anime_features.head()

In [ ]:
user_id_counts = anime_features['user_id'].value_counts()
user_id_counts.describe()

*In order to consider only the reviewers of "trusted" members, the ones we are considering trustworthy are those who have reviewed a certain number (100 in our case) of animes, the rest of the reviewers may be dropped.*

In [ ]:
anime_features = anime_features[anime_features['user_id'].isin(user_id_counts[user_id_counts >= 100].index)]

Since the title text was not found to be clean we a function to clean the title names using regex:

In [ ]:
def text_cleaning(text):
    text = re.sub(r'&quot;', '', text)
    text = re.sub(r'.hack//', '', text)
    text = re.sub(r'&#039;', '', text)
    text = re.sub(r'A&#039;s', '', text)
    text = re.sub(r'I&#039;', 'I\'', text)
    text = re.sub(r'&amp;', 'and', text)
    return text

In [ ]:
anime_features['anime_title'] = anime_features['anime_title'].apply(text_cleaning)
anime_pivot = anime_features.pivot_table(index='anime_title', columns='user_id', values='user_rating').fillna(0)

# Collaborative Filtering
Collaborative filtering is a technique used in recommender systems to predict user preferences or make recommendations by identifying similarities and patterns among users' preferences. It assumes that users with similar preferences in the past are likely to have similar preferences in the future. Collaborative filtering can be performed using user-based or item-based approaches, where user-based focuses on finding similar users to make recommendations, while item-based focuses on finding similar items. By leveraging these similarities, collaborative filtering helps users discover new items or content based on the preferences and behaviors of similar users, improving the overall user experience and engagement.

## Cosine Similarity using KNN

Cosine similarity is a measure of similarity between two non-zero vectors of an inner product space. In the context of recommendation systems, cosine similarity is often used to determine how similar two items or users are based on their feature vectors.


In [ ]:
anime_data["Name"] = anime_data["Name"].apply(text_cleaning)

Sparse matrix was created to optimize the memory usage and computational efficiency of the model. The user-anime matrix can be very large and the majority of the entries are likely to be zeros, which means that they don't contribute to the similarity computations. By converting the matrix to a sparse format, we can represent it using less memory, and perform computations only on the non-zero entries. This can significantly speed up the model training and recommendation generation processes.

In [ ]:
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

anime_matrix = csr_matrix(anime_pivot.values)

model_knn = NearestNeighbors(metric='cosine', algorithm='brute')

model_knn.fit(anime_matrix)

In [ ]:
anime_title = np.random.choice(anime_pivot.index)

print(f"Randomly selected anime title: {anime_title} \n")

query_index = anime_pivot.index.get_loc(anime_title)

distances, indices = model_knn.kneighbors(anime_pivot.iloc[query_index, :].values.reshape(1, -1), n_neighbors=6)

print(f"Recommendations for {anime_pivot.index[query_index]}:\n")

for i, (distance, index) in enumerate(zip(distances.flatten()[1:], indices.flatten()[1:])):
    print(f"{i+1}: {anime_pivot.index[index]}, with distance of {distance}")
    

In [ ]:
def give_rec_knn(anime_title = np.random.choice(anime_pivot.index), anime_pivot=anime_pivot):
    
    print(f"Selected anime title: {anime_title} \n")

    query_index = anime_pivot.index.get_loc(anime_title)

    distances, indices = model_knn.kneighbors(anime_pivot.iloc[query_index, :].values.reshape(1, -1), n_neighbors=6)

    print(f"Recommendations for {anime_pivot.index[query_index]}:\n")

    for i, (distance, index) in enumerate(zip(distances.flatten()[1:], indices.flatten()[1:])):
        print(f"{i+1}: {anime_pivot.index[index]}, with distance of {distance}")

In [ ]:
give_rec_knn("Steins;Gate")
give_rec_knn()

## Content Based Filtering
Content-based filtering is a recommendation system technique that recommends items based on their intrinsic features or attributes. It identifies items similar to the ones the user has shown interest in and recommends them. For example, a movie recommendation system might recommend other movies with similar genres, actors, directors, or plot themes to those previously watched by the user. Content-based filtering does not rely on the preferences of other users and can work well for new or niche items with little user data. However, it may suffer from limited diversity in recommendations and inability to capture serendipitous recommendations.

## TF-IDF Vectorizer
TF-IDF stands for Term Frequency-Inverse Document Frequency.<br>
TF-IDF vectorizer is a specific implementation of the TF-IDF technique. It is a commonly used technique in natural language processing to convert a collection of text documents into numerical feature vectors, which can be used for machine learning tasks like text classification or clustering.
* It first counts the number of occurrences of each word (term) in each document (text) in the collection, and then * calculates a weight for each term based on how frequently it appears across all documents. 
* The weight is higher for terms that appear frequently in a particular document, but not so much in other documents. 
* The weight is also higher for terms that appear less frequently across all documents.

This helps to give more importance to words that are relevant to a specific document and less importance to common words that are not specific to any document.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfv = TfidfVectorizer(min_df=3, max_features=None, strip_accents='unicode', analyzer='word', token_pattern=r'\w{1,}', ngram_range=(1, 3), stop_words='english')

anime_data['Genres'] = anime_data['Genres'].fillna('')

genres_str = anime_data['Genres'].str.split(',').astype(str)

tfv_matrix = tfv.fit_transform(genres_str)

print(tfv_matrix.shape)

The sigmoid kernel is a type of kernel function used in machine learning for non-linear classification and regression. It maps the data into a higher-dimensional space and computes the dot product between two data points in that space. 
* The sigmoid_kernel is a similarity function that computes the sigmoid kernel between two input feature vectors.

* It is commonly used in machine learning for non-linear classification and regression tasks.

* The sigmoid kernel function takes two feature vectors as input and computes a value between 0 and 1, where 1 indicates a high degree of similarity between the two vectors and 0 indicates no similarity.

* The sigmoid kernel function applies the sigmoid function to the dot product of the two feature vectors, which transforms the dot product into a value between 0 and 1.

* The `sigmoid_kernel` is being used to compute the similarity between anime genres based on their TF-IDF feature vectors, which can be used for content-based recommendation systems.

In [ ]:
from sklearn.metrics.pairwise import sigmoid_kernel

sig = sigmoid_kernel(tfv_matrix, tfv_matrix)

In [ ]:
indices = pd.Series(anime_data.index, index=anime_data['Name'])

indices = indices.drop_duplicates()

In [ ]:
def give_rec_cbf(title, sig=sig):
    idx = indices[title]
    
    sig_scores = list(enumerate(sig[idx]))
    
    sig_scores = sorted(sig_scores, key=lambda x: x[1], reverse=True)
    
    anime_indices = [i[0] for i in sig_scores[1: 11]]
    
    top_anime = pd.DataFrame({
        'Anime name': anime_data['Name'].iloc[anime_indices].values,
        'Rating': anime_data['Score'].iloc[anime_indices].values
    })
    return top_anime

In [ ]:
give_rec_cbf('One Piece')